** Notes: **
* This file parses the twitter user geolocation into a US State-county format.
* **Input file** is ../2-downloadGeolocation/out-userGeolocations/twitter-users-jan2018-6nov2018.csv
* **output file** is ./output/parsed-geoloc.csv

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import string
import unidecode
from tqdm import tqdm
from datetime import datetime

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 50)

## Load in geo data
Location data files used:
    * iso-country-codes.csv
    * us-counties.csv
    * uscitiesv1.4.csv

### Country codes

In [ ]:
geoDataFolder = os.path.join('.', 'countriesCitiesData')

fileCountryCodes = os.path.join(geoDataFolder, 'iso-country-codes.csv')
dfCountryCodes = pd.read_csv(fileCountryCodes)
# remove US
dfCountryCodes = dfCountryCodes.drop(index=[240,241])

countries = dfCountryCodes['country'].tolist()
alpha2 = dfCountryCodes['alpha2'].tolist()
alpha3 = dfCountryCodes['alpha3'].tolist()

In [ ]:
countriesToAdd = ['england', 'espana', 'scotland', 'northern ireland', 'nederland']
countries = countries + countriesToAdd

### US city counties list

In [ ]:
dropStates = ['american samoa', 
              'federated states of micronesia', 
              'guam', 
              'marshall islands', 
              'northern mariana islands', 
              'palau', 
              'puerto rico', 
              'us armed forces europe', 
              'us armed forces pacific', 
              'virgin islands', 
              'washington, d.c.']

fileCountiesCities = os.path.join(geoDataFolder, 'us-counties.csv')
dfCountiesCities = pd.read_csv(fileCountiesCities)

dfCountiesCities.drop(labels=['city alias'], axis=1, inplace=True)
dfCountiesCities.drop_duplicates(subset=['city', 'state short', 'state full', 'county'], 
                                 keep='first', 
                                 inplace=True)
dfCountiesCities.reset_index(drop=True, inplace=True)

dfCountiesCities = dfCountiesCities[dfCountiesCities['state full'].notnull()]

dfCountiesCities = dfCountiesCities[~dfCountiesCities['state full'].isin(dropStates)]

setStates = set(dfCountiesCities['state full'].tolist())

### Simple maps data

In [ ]:
fileSimpleMaps = os.path.join(geoDataFolder, 'uscitiesv1.4.csv')
dfSimpleMaps = pd.read_csv(fileSimpleMaps)
dfSimpleMaps['city'] = dfSimpleMaps['city'].str.lower()
dfSimpleMaps['state short'] = dfSimpleMaps['state_id'].str.lower()
dfSimpleMaps['state full'] = dfSimpleMaps['state_name'].str.lower()
dfSimpleMaps['county_name'] = dfSimpleMaps['county_name'].str.lower()

dfSimpleMaps['city'] = dfSimpleMaps['city'].str.replace('saint', 'st')

In [ ]:
dfSimpleMapsInc = dfSimpleMaps[dfSimpleMaps['incorporated']==True]
dfSimpleMapsNotInc = dfSimpleMaps[dfSimpleMaps['incorporated']==False]

## Matching
* if no comma in location string:
    * [Step 1] check if string is unambiguously a non-US country:
        * if len(string) is 2:
            * check if string matches an non-US ISO alpha-2 code (e.g. ca as Canada)
        * if len(string) is 3:
            * check if string matches an non-US ISO alpha-3 code (e.g. can as Canada)
        * if len(string) > 3:
            * check if string is a non-US country name (e.g. Canada)
    * [Step 4] if not unambiguously a non-US country from Step 1:
        * try location as a city state (e.g. try new york as New York city in New York)
            * if still no match:
                * try as a city that can be uniquely identified (e.g. chicago can be uniquely identified as Chicago City in Illinois 
    * [Step 5] if still not identified as a US city-State from Step 4:
        * try to see if a comma is implied (e.g. philadelphia pa, with the comma implied between the two words, can then be recorded as Philadelphia City in Pennsylvania; and of course, the string pa philadelphia will also be recorded as Philadelphia City in Pennsylvania)
        
* if comma is present in location string:
    * [Step 2] check if one side of the comma can be unambiguously identified as a non-US country using country name (e.g. beunos aires, argentina should be filtered out)
    
    * [Step 3] if not unambiguously filtered out as non-US country in Step 2, check if one side is a US State and the other a US City
        * check if one side of comma has length of 2, if yes:
            * use that 2 string as a US State code (e.g. al as Alabama), and use the other side as the city
        * if not:
            * try 1 side as a US State, and the other as City, and vice versa
            
    * [Step 6] if not identified in Step 3, try one side as an indicator of the US (e.g. usa, us, united states, united states of america)
        * the other side is then treated as a city, state (e.g. new york, usa as New York city in New York); and of course usa, new york will also be treated as New York city in New York)
        * if still not identified:
            * try the other side as a city only (e.g. new england, usa is New England in North Dakota)

### Preamble

In [ ]:
inputFile = os.path.join('..', '2-downloadGeolocation', 'out-userGeolocations', 'twitter-users-jan2018-6nov2018.csv')

df = pd.read_csv(inputFile)
# change obs. no. 69719 to have loc_not_found == 1
df.set_value(69719, 'loc_not_found', 1)

df['loc_not_found'] = df['loc_not_found'].fillna(0)
df['user_not_found'] = df['user_not_found'].fillna(0)

df['loc_not_found'] = df['loc_not_found'].astype(int)
df['user_not_found'] = df['user_not_found'].astype(int)

# filter out those where user cant be found
df = df[(df['user_not_found']==0) & (df['loc_not_found']==0)]
df.reset_index(drop=True, inplace=True)

df['location'] = df['location'].str.lower()

###### Cleaning and parsing location string

In [ ]:
retainChar = string.ascii_lowercase + ',' + ' '

def cleanLocation(string):
    """ Cleans the location string
        1. unidecode string
        2. retain only ASCII lowercase characters
        3. remove 'county' from the string
        4. strip whitespaces on both ends
    
        Returns
        -------
        Str
    """
    
    string = unidecode.unidecode(string)
    string = ''.join(char for char in string if char in retainChar)

    string = string.replace('county', '')

    string = string.strip()        

    return string

df['location'] = df['location'].apply(cleanLocation)

###### shuffling

if True:
    df = df.sample(n=100, random_state=0).reset_index(drop=True)

In [ ]:
if False:
    df = df.sample(n=1000).reset_index(drop=True)

### Step 1 (about 8 secs)

1. if location has no comma, check if it's a country other than the US (e.g. canada should be filtered out)
2. check location matches country name, country ISO alpha-2 or country ISO alpha-3
3. using dfCountryCodes read from 'iso-country-codes.csv'

In [ ]:
for ix in tqdm(range(0, len(df))):
    notUS = False
    location = df.at[ix, 'location']
    if ',' not in location:
        if location in countries:
            notUS = True
        if location in alpha2:
            notUS = True
        if location in alpha3:
            notUS= True

    if notUS:
        df.at[ix, 'not us'] = 1
        df.at[ix, 'step'] = 'filtered out at step 1'
        df.at[ix, 'notes'] = 'unambiguously identified as a non-US country'

### Step 2 (about 11 secs)
1. if location has a comma, check if one of the side matches a country other than the US (e.g. beunos aires, argentina should be filtered out)
2. using only country full name
3. using dfCountryCodes read from 'iso-country-codes.csv'

In [ ]:
for ix in tqdm(range(0, len(df))):
    if type(df.at[ix, 'step']) == str:
        continue
    location = df.at[ix, 'location']
    if location.count(',') == 1:
        firstLoc, secondLoc = location.split(',')
            
        for loc in [firstLoc, secondLoc]:
            if loc.strip() in countries:
                df.at[ix, 'not us'] = 1
                df.at[ix, 'step'] = 'filtered out at step 2'
                df.at[ix, 'notes'] = 'unambiguously identified as a non-US country'

### Step 3 (about 18mins)
3. if location has a comma, check if one of the sides are only 2 letters long (for state abbreviations)
    * if yes, search by state short name and city
    * if no, search by state full and city
    * use unincorporated city data if cannot find using incorporated city data

In [ ]:
for ix in tqdm(range(0, len(df))):
    if type(df.get_value(ix, 'step')) == str: # location filtered out before
        continue
        
    location = df.at[ix, 'location']
    
    if location.count(',') != 1:
        continue
        
    boolStateShort = False
    
    first, second = location.split(',')
    first = first.strip()
    second = second.strip()
    
    if (len(first) == 2) and (len(second) == 2):
        df.at[ix, 'step'] = 'filtered out at step 3'
        df.at[ix, 'notes'] = 'both side of comma has only 2 alphanumeric symbols'
    else:
        if len(first) == 2:
            boolStateShort = True
            stateShort = first
            city = second
        if len(second) == 2:
            boolStateShort = True
            stateShort = second
            city = first
            
        if boolStateShort:
            _dfState = (dfSimpleMapsInc[(dfSimpleMapsInc['state short'] == stateShort) 
                                        & (dfSimpleMapsInc['city'] == city)]).reset_index()
            
            if len(_dfState) == 1: 
                df.at[ix, 'notes'] = 'unambiguously mapped city, state to county' 
                df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
                df.at[ix, 'state short'] = _dfState.at[0, 'state short']
                df.at[ix, 'state full'] = _dfState.at[0, 'state full']

            if len(_dfState) == 0:
                # then try the unincorporated data
                _dfState = (dfSimpleMapsNotInc[(dfSimpleMapsNotInc['state short'] == stateShort) 
                                        & (dfSimpleMapsNotInc['city'] == city)]).reset_index()
                if len(_dfState) == 1: 
                    df.at[ix, 'notes'] = 'unambiguously mapped unincorporated city, state to county' 
                    df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
                    df.at[ix, 'state short'] = _dfState.at[0, 'state short']
                    df.at[ix, 'state full'] = _dfState.at[0, 'state full']
                else:
                    df.at[ix, 'notes'] = 'cannot get a city-state or county-city match'       
                    df.at[ix, 'us but not match'] = 1
                    df.at[ix, 'step'] = 'filtered out at step 3'
                    continue
                
        if not boolStateShort:
            # check which side of the comma is the state name in full
            if first in setStates:
                state = first
                city = second
            elif second in setStates:
                state = second
                city = first
            else:
                df.at[ix, 'notes'] = 'neither side of comma is a US state'
                df.at[ix, 'step'] = 'filtered out at step 3'
                continue
                
            
            _dfState = (dfSimpleMapsInc[(dfSimpleMapsInc['state full'] == state) 
                                        & (dfSimpleMapsInc['city'] == city)]).reset_index()            


            if len(_dfState) == 1:
                df.at[ix, 'notes'] = 'unambiguously mapped city, state to county'                  
                df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
                df.at[ix, 'state short'] = _dfState.at[0, 'state short']
                df.at[ix, 'state full'] = _dfState.at[0, 'state full']

            if len(_dfState) == 0:
                # then try the unincorporated data
                _dfState = (dfSimpleMapsNotInc[(dfSimpleMapsNotInc['state short'] == state) 
                                        & (dfSimpleMapsNotInc['city'] == city)]).reset_index()
                if len(_dfState) == 1: 
                    df.at[ix, 'notes'] = 'unambiguously mapped unincorporated city, state to county' 
                    df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
                    df.at[ix, 'state short'] = _dfState.at[0, 'state short']
                    df.at[ix, 'state full'] = _dfState.at[0, 'state full']
                else:
                    df.at[ix, 'notes'] = 'cannot get a city-state or county-city match'       
                    df.at[ix, 'us but not match'] = 1
                    df.at[ix, 'step'] = 'filtered out at step 3'

### Step 4 (about 30 mins)
1. if no comma in location, and it is not unambiguously a non-US country
    * try location as city state (e.g. new york as New York city in New York)
    * or try as city (e.g. chicago can be uniquely identified as chicago city in Illinois)

In [ ]:
for ix in tqdm(range(0, len(df))):
    location = df.at[ix, 'location']
    
    if location.count(',') != 0:
        continue
    
    if location in countries:
        continue

    # try as city, state (e.g. New York city in New York)                
    _dfState = (dfSimpleMapsInc[(dfSimpleMapsInc['state full'] == location) 
                                        & (dfSimpleMapsInc['city'] == location)]).reset_index() 
    
    if len(_dfState) == 1:
        df.at[ix, 'notes'] = 'entire location as city, state' 
        df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
        df.at[ix, 'state short'] = _dfState.at[0, 'state short']
        df.at[ix, 'state full'] = _dfState.at[0, 'state full']       
    
    else: # try as city (e.g. New England can be uniquely identified as )
                
        if location in setStates:
            continue
    
        _dfState = (dfSimpleMapsInc[(dfSimpleMapsInc['city'] == location)]).reset_index() 
        if len(_dfState) == 1:
            df.at[ix, 'notes'] = 'entire location as city' 
            df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
            df.at[ix, 'state short'] = _dfState.at[0, 'state short']
            df.at[ix, 'state full'] = _dfState.at[0, 'state full']       
        

### Step 5 (about 1hr 20mins)

* if no comma, look for location of the type city state w/o comma (e.g. philadelphia pa)

In [ ]:
for ix in tqdm(range(0, len(df))):
    location = df.at[ix, 'location']
    
    if location.count(',') != 0:
        continue
    
    if location in countries:
        continue
        
    listLocation = location.split(' ')
    
    # try as 'state (id) city' (e.g. pa philadelphia)
    city = listLocation.pop().strip()
    state = ' '.join(term for term in listLocation).strip()
    
    _dfState = (dfSimpleMapsInc[(dfSimpleMapsInc['state short'] == state) 
                                        & (dfSimpleMapsInc['city'] == city)]).reset_index()   
    
    if len(_dfState) == 1:
        df.at[ix, 'notes'] = 'parsed without comma as city state' 
        df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
        df.at[ix, 'state short'] = _dfState.at[0, 'state short']
        df.at[ix, 'state full'] = _dfState.at[0, 'state full']       
        
    else: # try as 'state (name) city' (e.g. pennsylvania philadelphia)
        _dfState = (dfSimpleMapsInc[(dfSimpleMapsInc['state full'] == state) 
                                        & (dfSimpleMapsInc['city'] == city)]).reset_index()   
    
        if len(_dfState)  == 1:
            df.at[ix, 'notes'] = 'parsed without comma as city state' 
            df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
            df.at[ix, 'state short'] = _dfState.at[0, 'state short']
            df.at[ix, 'state full'] = _dfState.at[0, 'state full']    
            
    # try as 'city state(id)' (e.g. pennsylvania philadelphia)
    listLocation = location.split(' ')
    listLocation.reverse()
    city = listLocation.pop().strip()
    state = ' '.join(term for term in listLocation).strip()
    
    _dfState = (dfSimpleMapsInc[(dfSimpleMapsInc['state short'] == state) 
                                        & (dfSimpleMapsInc['city'] == city)]).reset_index()   
    
    if len(_dfState) == 1:
        df.at[ix, 'notes'] = 'parsed without comma as city state' 
        df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
        df.at[ix, 'state short'] = _dfState.at[0, 'state short']
        df.at[ix, 'state full'] = _dfState.at[0, 'state full']       
        
    else: # try as 'state (name) city' (e.g. pennsylvania philadelphia)
        _dfState = (dfSimpleMapsInc[(dfSimpleMapsInc['state full'] == state) 
                                        & (dfSimpleMapsInc['city'] == city)]).reset_index()   
    
        if len(_dfState)  == 1:
            df.at[ix, 'notes'] = 'parsed without comma as city state' 
            df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
            df.at[ix, 'state short'] = _dfState.at[0, 'state short']
            df.at[ix, 'state full'] = _dfState.at[0, 'state full']    
                

### Step 6 (about 7 mins)
* if one of the sides of a comma refers to the US (e.g. usa, us, united states of america)
    * use the other side as a city, state (e.g. new york, usa as new york city in new york)
    * or use the other side as a city only (e.g. new england, usa as new england
    

In [ ]:
for ix in tqdm(range(0, len(df))):
    location = df.at[ix, 'location']
    
    if location.count(',') != 1:
        continue
        
    locContainsUSA = False
    
    if 'usa' in location:
        locContainsUSA =True
    if 'united states' in location:
        locContainsUSA = True
    if 'united states of america' in location:
        locContainsUSA = True
    
    if locContainsUSA:
        # try LEFT side of comma as a city, state (name) (e.g. new york, usa as new york in new york)
        loc = location.split(',')[0].strip()
        
        _dfState = (dfSimpleMapsInc[(dfSimpleMapsInc['state full'] == loc) 
                                        & (dfSimpleMapsInc['city'] == loc)]).reset_index() 
        
        if len(_dfState) == 1:
            df.at[ix, 'notes'] = 'one side of comma as a US city/state' 
            df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
            df.at[ix, 'state short'] = _dfState.at[0, 'state short']
            df.at[ix, 'state full'] = _dfState.at[0, 'state full']   
            
        # try RIGHT side of comma as a city, state (name) (e.g. usa, new york as new york in new york)
        loc = location.split(',')[1].strip()
        
        _dfState = (dfSimpleMapsInc[(dfSimpleMapsInc['state full'] == loc) 
                                        & (dfSimpleMapsInc['city'] == loc)]).reset_index() 
        
        if len(_dfState) == 1:
            df.at[ix, 'notes'] = 'one side of comma as a US city/state' 
            df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
            df.at[ix, 'state short'] = _dfState.at[0, 'state short']
            df.at[ix, 'state full'] = _dfState.at[0, 'state full']   
            
        # try LEFT side of comma as a city (e.g. new england, usa as new england)
        loc = location.split(',')[0].strip()
        
        if loc in setStates: 
            # cannot determine if city or state (e.g. cannot determine if michigan, usa is michigan city in north dakota 
            # or michigan state)
            continue
        
        _dfState = (dfSimpleMapsInc[dfSimpleMapsInc['city'] == loc]).reset_index() 
        
        if len(_dfState) == 1:
            df.at[ix, 'notes'] = 'one side of comma as a US city' 
            df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
            df.at[ix, 'state short'] = _dfState.at[0, 'state short']
            df.at[ix, 'state full'] = _dfState.at[0, 'state full']       
            
        # try RIGHT side of comma as a city (e.g. usa, new england as new england)
        loc = location.split(',')[1].strip()
        
        _dfState = (dfSimpleMapsInc[dfSimpleMapsInc['city'] == loc]).reset_index() 
        
        if len(_dfState) == 1:
            df.at[ix, 'notes'] = 'one side of comma as a US city' 
            df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
            df.at[ix, 'state short'] = _dfState.at[0, 'state short']
            df.at[ix, 'state full'] = _dfState.at[0, 'state full']              

In [ ]:
df.to_csv(os.path.join('output', 'parsed-geoloc.csv'), 
                       index=False)

## Check

In [ ]:
a = df[df['primary county'].isnull()]

a.head(200)

## Matching of post-election tweets
* if no comma in location string:
    * [Step 1] check if string is unambiguously a non-US country:
        * if len(string) is 2:
            * check if string matches an non-US ISO alpha-2 code (e.g. ca as Canada)
        * if len(string) is 3:
            * check if string matches an non-US ISO alpha-3 code (e.g. can as Canada)
        * if len(string) > 3:
            * check if string is a non-US country name (e.g. Canada)
    * [Step 4] if not unambiguously a non-US country from Step 1:
        * try location as a city state (e.g. try new york as New York city in New York)
            * if still no match:
                * try as a city that can be uniquely identified (e.g. chicago can be uniquely identified as Chicago City in Illinois 
    * [Step 5] if still not identified as a US city-State from Step 4:
        * try to see if a comma is implied (e.g. philadelphia pa, with the comma implied between the two words, can then be recorded as Philadelphia City in Pennsylvania; and of course, the string pa philadelphia will also be recorded as Philadelphia City in Pennsylvania)
        
* if comma is present in location string:
    * [Step 2] check if one side of the comma can be unambiguously identified as a non-US country using country name (e.g. beunos aires, argentina should be filtered out)
    
    * [Step 3] if not unambiguously filtered out as non-US country in Step 2, check if one side is a US State and the other a US City
        * check if one side of comma has length of 2, if yes:
            * use that 2 string as a US State code (e.g. al as Alabama), and use the other side as the city
        * if not:
            * try 1 side as a US State, and the other as City, and vice versa
            
    * [Step 6] if not identified in Step 3, try one side as an indicator of the US (e.g. usa, us, united states, united states of america)
        * the other side is then treated as a city, state (e.g. new york, usa as New York city in New York); and of course usa, new york will also be treated as New York city in New York)
        * if still not identified:
            * try the other side as a city only (e.g. new england, usa is New England in North Dakota)

### Preamble

In [ ]:
inputFile = os.path.join('..', '2-downloadGeolocation', 'out-userGeolocations', 'post-election-twitter-users.csv')

df = pd.read_csv(inputFile)

df['loc_not_found'] = df['loc_not_found'].fillna(0)
df['user_not_found'] = df['user_not_found'].fillna(0)

df['loc_not_found'] = df['loc_not_found'].astype(int)
df['user_not_found'] = df['user_not_found'].astype(int)

# filter out those where user cant be found
df = df[(df['user_not_found']==0) & (df['loc_not_found']==0)]
df.reset_index(drop=True, inplace=True)

df['location'] = df['location'].str.lower()

df = df[df['location'].notnull()]

df.reset_index(drop=True, inplace=True)

###### Cleaning and parsing location string

In [ ]:
retainChar = string.ascii_lowercase + ',' + ' '

def cleanLocation(string):
    """ Cleans the location string
        1. unidecode string
        2. retain only ASCII lowercase characters
        3. remove 'county' from the string
        4. strip whitespaces on both ends
    
        Returns
        -------
        Str
    """
    
    string = unidecode.unidecode(string)
    string = ''.join(char for char in string if char in retainChar)

    string = string.replace('county', '')

    string = string.strip()        

    return string

df['location'] = df['location'].apply(cleanLocation)

###### shuffling

if True:
    df = df.sample(n=100, random_state=0).reset_index(drop=True)

In [ ]:
if False:
    df = df.sample(n=1000).reset_index(drop=True)

### Step 1 (about 8 secs)

1. if location has no comma, check if it's a country other than the US (e.g. canada should be filtered out)
2. check location matches country name, country ISO alpha-2 or country ISO alpha-3
3. using dfCountryCodes read from 'iso-country-codes.csv'

In [ ]:
for ix in tqdm(range(0, len(df))):
    notUS = False
    location = df.at[ix, 'location']
    if ',' not in location:
        if location in countries:
            notUS = True
        if location in alpha2:
            notUS = True
        if location in alpha3:
            notUS= True

    if notUS:
        df.at[ix, 'not us'] = 1
        df.at[ix, 'step'] = 'filtered out at step 1'
        df.at[ix, 'notes'] = 'unambiguously identified as a non-US country'

### Step 2 (about 11 secs)
1. if location has a comma, check if one of the side matches a country other than the US (e.g. beunos aires, argentina should be filtered out)
2. using only country full name
3. using dfCountryCodes read from 'iso-country-codes.csv'

In [ ]:
for ix in tqdm(range(0, len(df))):
    if type(df.at[ix, 'step']) == str:
        continue
    location = df.at[ix, 'location']
    if location.count(',') == 1:
        firstLoc, secondLoc = location.split(',')
            
        for loc in [firstLoc, secondLoc]:
            if loc.strip() in countries:
                df.at[ix, 'not us'] = 1
                df.at[ix, 'step'] = 'filtered out at step 2'
                df.at[ix, 'notes'] = 'unambiguously identified as a non-US country'

### Step 3 (about 18mins)
3. if location has a comma, check if one of the sides are only 2 letters long (for state abbreviations)
    * if yes, search by state short name and city
    * if no, search by state full and city
    * use unincorporated city data if cannot find using incorporated city data

In [ ]:
for ix in tqdm(range(0, len(df))):
    if type(df.get_value(ix, 'step')) == str: # location filtered out before
        continue
        
    location = df.at[ix, 'location']
    
    if location.count(',') != 1:
        continue
        
    boolStateShort = False
    
    first, second = location.split(',')
    first = first.strip()
    second = second.strip()
    
    if (len(first) == 2) and (len(second) == 2):
        df.at[ix, 'step'] = 'filtered out at step 3'
        df.at[ix, 'notes'] = 'both side of comma has only 2 alphanumeric symbols'
    else:
        if len(first) == 2:
            boolStateShort = True
            stateShort = first
            city = second
        if len(second) == 2:
            boolStateShort = True
            stateShort = second
            city = first
            
        if boolStateShort:
            _dfState = (dfSimpleMapsInc[(dfSimpleMapsInc['state short'] == stateShort) 
                                        & (dfSimpleMapsInc['city'] == city)]).reset_index()
            
            if len(_dfState) == 1: 
                df.at[ix, 'notes'] = 'unambiguously mapped city, state to county' 
                df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
                df.at[ix, 'state short'] = _dfState.at[0, 'state short']
                df.at[ix, 'state full'] = _dfState.at[0, 'state full']

            if len(_dfState) == 0:
                # then try the unincorporated data
                _dfState = (dfSimpleMapsNotInc[(dfSimpleMapsNotInc['state short'] == stateShort) 
                                        & (dfSimpleMapsNotInc['city'] == city)]).reset_index()
                if len(_dfState) == 1: 
                    df.at[ix, 'notes'] = 'unambiguously mapped unincorporated city, state to county' 
                    df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
                    df.at[ix, 'state short'] = _dfState.at[0, 'state short']
                    df.at[ix, 'state full'] = _dfState.at[0, 'state full']
                else:
                    df.at[ix, 'notes'] = 'cannot get a city-state or county-city match'       
                    df.at[ix, 'us but not match'] = 1
                    df.at[ix, 'step'] = 'filtered out at step 3'
                    continue
                
        if not boolStateShort:
            # check which side of the comma is the state name in full
            if first in setStates:
                state = first
                city = second
            elif second in setStates:
                state = second
                city = first
            else:
                df.at[ix, 'notes'] = 'neither side of comma is a US state'
                df.at[ix, 'step'] = 'filtered out at step 3'
                continue
                
            
            _dfState = (dfSimpleMapsInc[(dfSimpleMapsInc['state full'] == state) 
                                        & (dfSimpleMapsInc['city'] == city)]).reset_index()            


            if len(_dfState) == 1:
                df.at[ix, 'notes'] = 'unambiguously mapped city, state to county'                  
                df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
                df.at[ix, 'state short'] = _dfState.at[0, 'state short']
                df.at[ix, 'state full'] = _dfState.at[0, 'state full']

            if len(_dfState) == 0:
                # then try the unincorporated data
                _dfState = (dfSimpleMapsNotInc[(dfSimpleMapsNotInc['state short'] == state) 
                                        & (dfSimpleMapsNotInc['city'] == city)]).reset_index()
                if len(_dfState) == 1: 
                    df.at[ix, 'notes'] = 'unambiguously mapped unincorporated city, state to county' 
                    df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
                    df.at[ix, 'state short'] = _dfState.at[0, 'state short']
                    df.at[ix, 'state full'] = _dfState.at[0, 'state full']
                else:
                    df.at[ix, 'notes'] = 'cannot get a city-state or county-city match'       
                    df.at[ix, 'us but not match'] = 1
                    df.at[ix, 'step'] = 'filtered out at step 3'

### Step 4 (about 30 mins)
1. if no comma in location, and it is not unambiguously a non-US country
    * try location as city state (e.g. new york as New York city in New York)
    * or try as city (e.g. chicago can be uniquely identified as chicago city in Illinois)

In [ ]:
for ix in tqdm(range(0, len(df))):
    location = df.at[ix, 'location']
    
    if location.count(',') != 0:
        continue
    
    if location in countries:
        continue

    # try as city, state (e.g. New York city in New York)                
    _dfState = (dfSimpleMapsInc[(dfSimpleMapsInc['state full'] == location) 
                                        & (dfSimpleMapsInc['city'] == location)]).reset_index() 
    
    if len(_dfState) == 1:
        df.at[ix, 'notes'] = 'entire location as city, state' 
        df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
        df.at[ix, 'state short'] = _dfState.at[0, 'state short']
        df.at[ix, 'state full'] = _dfState.at[0, 'state full']       
    
    else: # try as city (e.g. New England can be uniquely identified as )
                
        if location in setStates:
            continue
    
        _dfState = (dfSimpleMapsInc[(dfSimpleMapsInc['city'] == location)]).reset_index() 
        if len(_dfState) == 1:
            df.at[ix, 'notes'] = 'entire location as city' 
            df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
            df.at[ix, 'state short'] = _dfState.at[0, 'state short']
            df.at[ix, 'state full'] = _dfState.at[0, 'state full']       
        

### Step 5 (about 1hr 20mins)

* if no comma, look for location of the type city state w/o comma (e.g. philadelphia pa)

In [ ]:
for ix in tqdm(range(0, len(df))):
    location = df.at[ix, 'location']
    
    if location.count(',') != 0:
        continue
    
    if location in countries:
        continue
        
    listLocation = location.split(' ')
    
    # try as 'state (id) city' (e.g. pa philadelphia)
    city = listLocation.pop().strip()
    state = ' '.join(term for term in listLocation).strip()
    
    _dfState = (dfSimpleMapsInc[(dfSimpleMapsInc['state short'] == state) 
                                        & (dfSimpleMapsInc['city'] == city)]).reset_index()   
    
    if len(_dfState) == 1:
        df.at[ix, 'notes'] = 'parsed without comma as city state' 
        df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
        df.at[ix, 'state short'] = _dfState.at[0, 'state short']
        df.at[ix, 'state full'] = _dfState.at[0, 'state full']       
        
    else: # try as 'state (name) city' (e.g. pennsylvania philadelphia)
        _dfState = (dfSimpleMapsInc[(dfSimpleMapsInc['state full'] == state) 
                                        & (dfSimpleMapsInc['city'] == city)]).reset_index()   
    
        if len(_dfState)  == 1:
            df.at[ix, 'notes'] = 'parsed without comma as city state' 
            df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
            df.at[ix, 'state short'] = _dfState.at[0, 'state short']
            df.at[ix, 'state full'] = _dfState.at[0, 'state full']    
            
    # try as 'city state(id)' (e.g. pennsylvania philadelphia)
    listLocation = location.split(' ')
    listLocation.reverse()
    city = listLocation.pop().strip()
    state = ' '.join(term for term in listLocation).strip()
    
    _dfState = (dfSimpleMapsInc[(dfSimpleMapsInc['state short'] == state) 
                                        & (dfSimpleMapsInc['city'] == city)]).reset_index()   
    
    if len(_dfState) == 1:
        df.at[ix, 'notes'] = 'parsed without comma as city state' 
        df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
        df.at[ix, 'state short'] = _dfState.at[0, 'state short']
        df.at[ix, 'state full'] = _dfState.at[0, 'state full']       
        
    else: # try as 'state (name) city' (e.g. pennsylvania philadelphia)
        _dfState = (dfSimpleMapsInc[(dfSimpleMapsInc['state full'] == state) 
                                        & (dfSimpleMapsInc['city'] == city)]).reset_index()   
    
        if len(_dfState)  == 1:
            df.at[ix, 'notes'] = 'parsed without comma as city state' 
            df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
            df.at[ix, 'state short'] = _dfState.at[0, 'state short']
            df.at[ix, 'state full'] = _dfState.at[0, 'state full']    
                

### Step 6 (about 7 mins)
* if one of the sides of a comma refers to the US (e.g. usa, us, united states of america)
    * use the other side as a city, state (e.g. new york, usa as new york city in new york)
    * or use the other side as a city only (e.g. new england, usa as new england
    

In [ ]:
for ix in tqdm(range(0, len(df))):
    location = df.at[ix, 'location']
    
    if location.count(',') != 1:
        continue
        
    locContainsUSA = False
    
    if 'usa' in location:
        locContainsUSA =True
    if 'united states' in location:
        locContainsUSA = True
    if 'united states of america' in location:
        locContainsUSA = True
    
    if locContainsUSA:
        # try LEFT side of comma as a city, state (name) (e.g. new york, usa as new york in new york)
        loc = location.split(',')[0].strip()
        
        _dfState = (dfSimpleMapsInc[(dfSimpleMapsInc['state full'] == loc) 
                                        & (dfSimpleMapsInc['city'] == loc)]).reset_index() 
        
        if len(_dfState) == 1:
            df.at[ix, 'notes'] = 'one side of comma as a US city/state' 
            df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
            df.at[ix, 'state short'] = _dfState.at[0, 'state short']
            df.at[ix, 'state full'] = _dfState.at[0, 'state full']   
            
        # try RIGHT side of comma as a city, state (name) (e.g. usa, new york as new york in new york)
        loc = location.split(',')[1].strip()
        
        _dfState = (dfSimpleMapsInc[(dfSimpleMapsInc['state full'] == loc) 
                                        & (dfSimpleMapsInc['city'] == loc)]).reset_index() 
        
        if len(_dfState) == 1:
            df.at[ix, 'notes'] = 'one side of comma as a US city/state' 
            df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
            df.at[ix, 'state short'] = _dfState.at[0, 'state short']
            df.at[ix, 'state full'] = _dfState.at[0, 'state full']   
            
        # try LEFT side of comma as a city (e.g. new england, usa as new england)
        loc = location.split(',')[0].strip()
        
        if loc in setStates: 
            # cannot determine if city or state (e.g. cannot determine if michigan, usa is michigan city in north dakota 
            # or michigan state)
            continue
        
        _dfState = (dfSimpleMapsInc[dfSimpleMapsInc['city'] == loc]).reset_index() 
        
        if len(_dfState) == 1:
            df.at[ix, 'notes'] = 'one side of comma as a US city' 
            df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
            df.at[ix, 'state short'] = _dfState.at[0, 'state short']
            df.at[ix, 'state full'] = _dfState.at[0, 'state full']       
            
        # try RIGHT side of comma as a city (e.g. usa, new england as new england)
        loc = location.split(',')[1].strip()
        
        _dfState = (dfSimpleMapsInc[dfSimpleMapsInc['city'] == loc]).reset_index() 
        
        if len(_dfState) == 1:
            df.at[ix, 'notes'] = 'one side of comma as a US city' 
            df.at[ix, 'primary county'] = _dfState.at[0, 'county_name']
            df.at[ix, 'state short'] = _dfState.at[0, 'state short']
            df.at[ix, 'state full'] = _dfState.at[0, 'state full']              

In [ ]:
df.head(50)

In [ ]:
df.to_csv(os.path.join('output', 'post-election-parsed-geoloc.csv'), 
                       index=False)